In [1]:
# Reinitialize environment and continue evaluation
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm

os.chdir('/home/smallyan/eval_agent')
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects2/chai-lab/shared_models/hub'

print(f"Working directory: {os.getcwd()}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Working directory: /home/smallyan/eval_agent
CUDA available: True
Using device: cuda


In [2]:
# Load notebook to continue evaluation
notebook_path = '/net/scratch2/smallyan/open_question/sarcasm_claude_2026-01-05_00-24-43/notebooks/2026-01-05-00-24_SarcasmCircuitAnalysis.ipynb'

with open(notebook_path, 'r') as f:
    notebook = json.load(f)

# Extract code cells
code_cells = []
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'code':
        source = ''.join(cell['source'])
        code_cells.append({
            'index': i,
            'source': source
        })

# Re-create evaluation results storage
code_cells_info = []
for j, cell_info in enumerate(code_cells):
    code_cells_info.append({
        'block_id': f"Cell_{j}",
        'notebook_idx': cell_info['index'],
        'source': cell_info['source'],
        'description': '',
        'runnable': None,
        'correct_implementation': None,
        'redundant': None,
        'irrelevant': None,
        'error_note': ''
    })

print(f"Total code blocks: {len(code_cells_info)}")

# Mark blocks 0-16 as already evaluated (based on prior session)
block_descriptions = [
    "Setup working directory",
    "Check CUDA and PyTorch availability",
    "Import necessary libraries",
    "Load GPT-2 Small model with HookedTransformer",
    "Create dataset of sarcastic vs literal sentences",
    "Tokenize and examine sentences",
    "Analyze next-token predictions for sarcastic sentences",
    "Analyze next-token predictions for literal sentences",
    "Create cleaner experimental setup with sentiment probes",
    "Collect activations for minimal pairs",
    "Analyze residual stream divergence across layers",
    "Visualize the divergence pattern",
    "Define sentiment logit diff function and get baseline",
    "Define head patching function",
    "Patch all attention heads and collect results",
    "Visualize head patching effects",
    "Patch MLPs",
]

for i in range(17):
    code_cells_info[i]['description'] = block_descriptions[i]
    code_cells_info[i]['runnable'] = 'Y'
    code_cells_info[i]['correct_implementation'] = 'Y'
    code_cells_info[i]['redundant'] = 'N'
    code_cells_info[i]['irrelevant'] = 'N'

print("Blocks 0-16 marked as evaluated from prior session")

Total code blocks: 31
Blocks 0-16 marked as evaluated from prior session


In [3]:
# Load model to continue evaluation
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Model loaded: {model.cfg.model_name}")

# Define necessary functions and data
positive_words = [" happy", " good", " great", " excited", " wonderful", " amazing", " pleased", " glad", " thrilled", " joyful"]
negative_words = [" bad", " sad", " terrible", " awful", " frustrated", " annoyed", " upset", " angry", " miserable", " disappointed"]

positive_tokens = [model.to_tokens(w, prepend_bos=False)[0, 0].item() for w in positive_words]
negative_tokens = [model.to_tokens(w, prepend_bos=False)[0, 0].item() for w in negative_words]

test_sarcastic = "Perfect, I lost my wallet. I feel"
test_literal = "Perfect, I found some money. I feel"

def get_sentiment_logit_diff(prompt, positive_tokens, negative_tokens):
    """Get the difference between positive and negative sentiment logits."""
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)
    last_logits = logits[0, -1, :]
    
    pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
    neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
    
    return (pos_logit - neg_logit).item()

def patch_head_and_measure(sarcastic_prompt, literal_prompt, layer, head, positive_tokens, negative_tokens):
    """Patch attention head output from literal into sarcastic and measure change."""
    tokens_lit = model.to_tokens(literal_prompt)
    _, cache_lit = model.run_with_cache(tokens_lit)
    
    def patch_hook(activation, hook):
        activation[0, -1, head, :] = cache_lit[hook.name][0, -1, head, :]
        return activation
    
    tokens_sarc = model.to_tokens(sarcastic_prompt)
    hook_name = f"blocks.{layer}.attn.hook_z"
    
    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            tokens_sarc,
            fwd_hooks=[(hook_name, patch_hook)]
        )
    
    last_logits = patched_logits[0, -1, :]
    pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
    neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
    
    return (pos_logit - neg_logit).item()

def patch_mlp_and_measure(sarcastic_prompt, literal_prompt, layer, positive_tokens, negative_tokens):
    """Patch MLP output from literal into sarcastic and measure change."""
    tokens_lit = model.to_tokens(literal_prompt)
    _, cache_lit = model.run_with_cache(tokens_lit)
    
    def patch_hook(activation, hook):
        activation[0, -1, :] = cache_lit[hook.name][0, -1, :]
        return activation
    
    tokens_sarc = model.to_tokens(sarcastic_prompt)
    hook_name = f"blocks.{layer}.hook_mlp_out"
    
    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            tokens_sarc,
            fwd_hooks=[(hook_name, patch_hook)]
        )
    
    last_logits = patched_logits[0, -1, :]
    pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
    neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
    
    return (pos_logit - neg_logit).item()

baseline_sarc = get_sentiment_logit_diff(test_sarcastic, positive_tokens, negative_tokens)
print(f"Baseline sarcastic sentiment diff: {baseline_sarc:.4f}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Model loaded: gpt2


Baseline sarcastic sentiment diff: -1.8634


In [4]:
# Block 17: Validate across multiple sentence pairs
print("=" * 60)
print("BLOCK 17: Validate across multiple sentence pairs")
print("=" * 60)

try:
    print("Validating across multiple sentence pairs...")

    all_head_effects = []
    all_mlp_effects = []

    validation_pairs = [
        ("Oh great, another boring meeting. I feel", "Oh great, a wonderful surprise. I feel"),
        ("Wonderful, more tedious paperwork. I feel", "Wonderful, a fantastic opportunity. I feel"),
        ("Perfect, I lost my wallet today. I feel", "Perfect, I found twenty dollars. I feel"),
        ("Fantastic, the car broke down. I feel", "Fantastic, the team won today. I feel"),
        ("Amazing, I have to work overtime. I feel", "Amazing, I got a raise today. I feel"),
    ]

    for sarc, lit in tqdm(validation_pairs):
        # Get baselines
        baseline_s = get_sentiment_logit_diff(sarc, positive_tokens, negative_tokens)
        
        # Patch heads
        pair_head_effects = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
        for layer in range(model.cfg.n_layers):
            for head in range(model.cfg.n_heads):
                result = patch_head_and_measure(sarc, lit, layer, head, positive_tokens, negative_tokens)
                pair_head_effects[layer, head] = result - baseline_s
        all_head_effects.append(pair_head_effects)
        
        # Patch MLPs
        pair_mlp_effects = []
        for layer in range(model.cfg.n_layers):
            result = patch_mlp_and_measure(sarc, lit, layer, positive_tokens, negative_tokens)
            pair_mlp_effects.append(result - baseline_s)
        all_mlp_effects.append(pair_mlp_effects)

    # Average effects
    avg_head_effects = np.mean(all_head_effects, axis=0)
    avg_mlp_effects = np.mean(all_mlp_effects, axis=0)

    print("\nTop attention heads (averaged across pairs):")
    flat_indices = np.argsort(avg_head_effects.flatten())[::-1]
    for idx in flat_indices[:10]:
        layer = idx // model.cfg.n_heads
        head = idx % model.cfg.n_heads
        effect = avg_head_effects[layer, head]
        print(f"  a{layer}.h{head}: {effect:.4f}")
    
    # Evaluation
    code_cells_info[17]['description'] = "Validate across multiple sentence pairs"
    code_cells_info[17]['runnable'] = 'Y'
    code_cells_info[17]['correct_implementation'] = 'Y'
    code_cells_info[17]['redundant'] = 'N'
    code_cells_info[17]['irrelevant'] = 'N'
    print("\n✓ Block 17 executed successfully")
except Exception as e:
    code_cells_info[17]['runnable'] = 'N'
    code_cells_info[17]['error_note'] = str(e)
    print(f"\n✗ Block 17 failed: {e}")

BLOCK 17: Validate across multiple sentence pairs
Validating across multiple sentence pairs...


  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:16<01:05, 16.46s/it]

 40%|████      | 2/5 [00:33<00:49, 16.57s/it]

 60%|██████    | 3/5 [00:49<00:32, 16.48s/it]

 80%|████████  | 4/5 [01:05<00:16, 16.36s/it]

100%|██████████| 5/5 [01:22<00:00, 16.50s/it]

100%|██████████| 5/5 [01:22<00:00, 16.48s/it]


Top attention heads (averaged across pairs):
  a9.h2: 0.4339
  a6.h7: 0.3473
  a7.h5: 0.2247
  a6.h3: 0.1896
  a8.h5: 0.1612
  a10.h4: 0.1610
  a9.h3: 0.1569
  a6.h4: 0.1178
  a9.h10: 0.1116
  a10.h9: 0.0854

✓ Block 17 executed successfully


In [5]:
# Block 18: Visualize validated results
print("=" * 60)
print("BLOCK 18: Visualize validated results")
print("=" * 60)

try:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Head effects
    im1 = axes[0].imshow(avg_head_effects, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
    plt.colorbar(im1, ax=axes[0], label='Patching Effect')
    axes[0].set_xlabel('Head', fontsize=12)
    axes[0].set_ylabel('Layer', fontsize=12)
    axes[0].set_title('Attention Head Patching Effects\n(Averaged across 5 pairs)', fontsize=12)
    axes[0].set_xticks(range(12))
    axes[0].set_yticks(range(12))

    # Mark important heads
    for layer in range(12):
        for head in range(12):
            if avg_head_effects[layer, head] > 0.15:
                axes[0].scatter(head, layer, marker='*', s=200, c='yellow', edgecolors='black')

    # MLP effects
    colors = ['red' if e < 0 else 'blue' for e in avg_mlp_effects]
    axes[1].barh(range(12), avg_mlp_effects, color=colors, edgecolor='black')
    axes[1].set_xlabel('Patching Effect', fontsize=12)
    axes[1].set_ylabel('Layer (MLP)', fontsize=12)
    axes[1].set_title('MLP Patching Effects\n(Averaged across 5 pairs)', fontsize=12)
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_yticks(range(12))
    axes[1].set_yticklabels([f'm{i}' for i in range(12)])

    plt.tight_layout()
    output_dir = '/net/scratch2/smallyan/open_question/sarcasm_claude_2026-01-05_00-24-43/results'
    plt.savefig(f'{output_dir}/validated_patching_effects.png', dpi=150)
    plt.close()

    print("Top MLPs (averaged across pairs):")
    for idx in np.argsort(avg_mlp_effects)[::-1][:5]:
        print(f"  m{idx}: {avg_mlp_effects[idx]:.4f}")
    
    # Evaluation
    code_cells_info[18]['description'] = "Visualize validated results"
    code_cells_info[18]['runnable'] = 'Y'
    code_cells_info[18]['correct_implementation'] = 'Y'
    code_cells_info[18]['redundant'] = 'N'
    code_cells_info[18]['irrelevant'] = 'N'
    print("\n✓ Block 18 executed successfully")
except Exception as e:
    code_cells_info[18]['runnable'] = 'N'
    code_cells_info[18]['error_note'] = str(e)
    print(f"\n✗ Block 18 failed: {e}")

BLOCK 18: Visualize validated results


Top MLPs (averaged across pairs):
  m9: 0.4209
  m10: 0.4106
  m8: 0.3455
  m6: 0.1495
  m7: 0.1124

✓ Block 18 executed successfully


In [6]:
# Block 19: Analyze attention patterns of key heads
print("=" * 60)
print("BLOCK 19: Analyze attention patterns of key heads")
print("=" * 60)

try:
    def get_attention_pattern(prompt, layer, head):
        """Get attention pattern for a specific head."""
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        
        # Get attention pattern: [batch, head, dest_pos, src_pos]
        attn_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]
        return attn_pattern[0, head, :, :].cpu().numpy()

    # Analyze key heads on sarcastic vs literal
    key_heads = [(9, 2), (6, 7), (7, 5), (6, 3), (8, 5)]

    test_sarc = "Oh great, another boring meeting"
    test_lit = "Oh great, a wonderful surprise"

    tokens_sarc = model.to_str_tokens(test_sarc)
    tokens_lit = model.to_str_tokens(test_lit)

    print(f"Sarcastic tokens: {tokens_sarc}")
    print(f"Literal tokens: {tokens_lit}")

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))

    for i, (layer, head) in enumerate(key_heads):
        # Sarcastic attention
        attn_sarc = get_attention_pattern(test_sarc, layer, head)
        im1 = axes[0, i].imshow(attn_sarc, cmap='Blues', vmin=0, vmax=1)
        axes[0, i].set_title(f'a{layer}.h{head} (Sarcastic)', fontsize=10)
        axes[0, i].set_xticks(range(len(tokens_sarc)))
        axes[0, i].set_xticklabels(tokens_sarc, rotation=45, ha='right', fontsize=8)
        axes[0, i].set_yticks(range(len(tokens_sarc)))
        axes[0, i].set_yticklabels(tokens_sarc, fontsize=8)
        
        # Literal attention
        attn_lit = get_attention_pattern(test_lit, layer, head)
        im2 = axes[1, i].imshow(attn_lit, cmap='Blues', vmin=0, vmax=1)
        axes[1, i].set_title(f'a{layer}.h{head} (Literal)', fontsize=10)
        axes[1, i].set_xticks(range(len(tokens_lit)))
        axes[1, i].set_xticklabels(tokens_lit, rotation=45, ha='right', fontsize=8)
        axes[1, i].set_yticks(range(len(tokens_lit)))
        axes[1, i].set_yticklabels(tokens_lit, fontsize=8)

    plt.tight_layout()
    output_dir = '/net/scratch2/smallyan/open_question/sarcasm_claude_2026-01-05_00-24-43/results'
    plt.savefig(f'{output_dir}/key_head_attention_patterns.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # Evaluation
    code_cells_info[19]['description'] = "Analyze attention patterns of key heads"
    code_cells_info[19]['runnable'] = 'Y'
    code_cells_info[19]['correct_implementation'] = 'Y'
    code_cells_info[19]['redundant'] = 'N'
    code_cells_info[19]['irrelevant'] = 'N'
    print("\n✓ Block 19 executed successfully")
except Exception as e:
    code_cells_info[19]['runnable'] = 'N'
    code_cells_info[19]['error_note'] = str(e)
    print(f"\n✗ Block 19 failed: {e}")

BLOCK 19: Analyze attention patterns of key heads
Sarcastic tokens: ['<|endoftext|>', 'Oh', ' great', ',', ' another', ' boring', ' meeting']
Literal tokens: ['<|endoftext|>', 'Oh', ' great', ',', ' a', ' wonderful', ' surprise']



✓ Block 19 executed successfully


In [7]:
# Block 20: Analyze what tokens each key head attends to from the last position
print("=" * 60)
print("BLOCK 20: Analyze attention from last token position")
print("=" * 60)

try:
    print("Attention from last token position for key heads:")
    print("=" * 70)

    for layer, head in key_heads:
        attn_sarc = get_attention_pattern(test_sarc, layer, head)
        attn_lit = get_attention_pattern(test_lit, layer, head)
        
        print(f"\na{layer}.h{head}:")
        print(f"  Sarcastic ('{test_sarc}'):")
        last_row_sarc = attn_sarc[-1, :]
        for i, (token, weight) in enumerate(zip(tokens_sarc, last_row_sarc)):
            if weight > 0.05:
                print(f"    -> '{token}': {weight:.3f}")
        
        print(f"  Literal ('{test_lit}'):")
        last_row_lit = attn_lit[-1, :]
        for i, (token, weight) in enumerate(zip(tokens_lit, last_row_lit)):
            if weight > 0.05:
                print(f"    -> '{token}': {weight:.3f}")
    
    # Evaluation
    code_cells_info[20]['description'] = "Analyze attention from last token position"
    code_cells_info[20]['runnable'] = 'Y'
    code_cells_info[20]['correct_implementation'] = 'Y'
    code_cells_info[20]['redundant'] = 'N'
    code_cells_info[20]['irrelevant'] = 'N'
    print("\n✓ Block 20 executed successfully")
except Exception as e:
    code_cells_info[20]['runnable'] = 'N'
    code_cells_info[20]['error_note'] = str(e)
    print(f"\n✗ Block 20 failed: {e}")

BLOCK 20: Analyze attention from last token position
Attention from last token position for key heads:

a9.h2:
  Sarcastic ('Oh great, another boring meeting'):
    -> '<|endoftext|>': 0.943
  Literal ('Oh great, a wonderful surprise'):
    -> '<|endoftext|>': 0.926



a6.h7:
  Sarcastic ('Oh great, another boring meeting'):
    -> '<|endoftext|>': 0.593
    -> ' boring': 0.129
    -> ' meeting': 0.211
  Literal ('Oh great, a wonderful surprise'):
    -> '<|endoftext|>': 0.810
    -> ' surprise': 0.079

a7.h5:
  Sarcastic ('Oh great, another boring meeting'):
    -> '<|endoftext|>': 0.550
    -> ' great': 0.058
    -> ',': 0.063
    -> ' boring': 0.197
    -> ' meeting': 0.074
  Literal ('Oh great, a wonderful surprise'):
    -> '<|endoftext|>': 0.670
    -> ' great': 0.069
    -> ',': 0.128
    -> ' surprise': 0.068



a6.h3:
  Sarcastic ('Oh great, another boring meeting'):
    -> '<|endoftext|>': 0.905
  Literal ('Oh great, a wonderful surprise'):
    -> '<|endoftext|>': 0.858

a8.h5:
  Sarcastic ('Oh great, another boring meeting'):
    -> '<|endoftext|>': 0.381
    -> ' great': 0.104
    -> ',': 0.123
    -> ' another': 0.190
    -> ' boring': 0.144
  Literal ('Oh great, a wonderful surprise'):
    -> '<|endoftext|>': 0.536
    -> ' great': 0.070
    -> ',': 0.089
    -> ' a': 0.095
    -> ' wonderful': 0.127

✓ Block 20 executed successfully


In [8]:
# Block 21: Define ablation functions
print("=" * 60)
print("BLOCK 21: Define ablation functions")
print("=" * 60)

try:
    candidate_heads = [(9, 2), (6, 7), (7, 5), (6, 3), (8, 5), (10, 4), (9, 3), (6, 4), (9, 10), (10, 9)]
    candidate_mlps = [6, 7, 8, 9, 10]

    def ablate_head_and_measure(prompt, layer, head, positive_tokens, negative_tokens):
        """Ablate (zero out) attention head output and measure change."""
        
        def ablate_hook(activation, hook):
            activation[0, :, head, :] = 0
            return activation
        
        tokens = model.to_tokens(prompt)
        hook_name = f"blocks.{layer}.attn.hook_z"
        
        with torch.no_grad():
            ablated_logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(hook_name, ablate_hook)]
            )
        
        last_logits = ablated_logits[0, -1, :]
        pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
        neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
        
        return (pos_logit - neg_logit).item()

    def ablate_mlp_and_measure(prompt, layer, positive_tokens, negative_tokens):
        """Ablate (zero out) MLP output and measure change."""
        
        def ablate_hook(activation, hook):
            activation[0, :, :] = 0
            return activation
        
        tokens = model.to_tokens(prompt)
        hook_name = f"blocks.{layer}.hook_mlp_out"
        
        with torch.no_grad():
            ablated_logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(hook_name, ablate_hook)]
            )
        
        last_logits = ablated_logits[0, -1, :]
        pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
        neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
        
        return (pos_logit - neg_logit).item()

    # Test on sarcastic prompts
    test_prompts = [
        "Perfect, I lost my wallet. I feel",
        "Oh great, another boring meeting. I feel",
        "Wonderful, more tedious paperwork. I feel",
    ]
    
    print("Testing ablation effects on sarcastic sentences...")
    print("(Positive change = sarcasm detection impaired)")
    
    for prompt in test_prompts:
        baseline = get_sentiment_logit_diff(prompt, positive_tokens, negative_tokens)
        ablated = ablate_head_and_measure(prompt, 6, 7, positive_tokens, negative_tokens)
        change = ablated - baseline
        print(f"\n'{prompt}'")
        print(f"  Baseline: {baseline:.4f}, After ablating a6.h7: {ablated:.4f}, Change: {change:+.4f}")
    
    # Evaluation
    code_cells_info[21]['description'] = "Define ablation functions"
    code_cells_info[21]['runnable'] = 'Y'
    code_cells_info[21]['correct_implementation'] = 'Y'
    code_cells_info[21]['redundant'] = 'N'
    code_cells_info[21]['irrelevant'] = 'N'
    print("\n✓ Block 21 executed successfully")
except Exception as e:
    code_cells_info[21]['runnable'] = 'N'
    code_cells_info[21]['error_note'] = str(e)
    print(f"\n✗ Block 21 failed: {e}")

BLOCK 21: Define ablation functions
Testing ablation effects on sarcastic sentences...
(Positive change = sarcasm detection impaired)

'Perfect, I lost my wallet. I feel'
  Baseline: -1.8634, After ablating a6.h7: -1.0986, Change: +0.7648

'Oh great, another boring meeting. I feel'
  Baseline: -1.6365, After ablating a6.h7: -1.3654, Change: +0.2711



'Wonderful, more tedious paperwork. I feel'
  Baseline: -1.2448, After ablating a6.h7: -0.8214, Change: +0.4233

✓ Block 21 executed successfully


In [9]:
# Block 22: Comprehensive ablation study
print("=" * 60)
print("BLOCK 22: Comprehensive ablation study")
print("=" * 60)

try:
    print("Comprehensive ablation study...")
    print("=" * 70)

    # Collect ablation effects across multiple prompts
    head_ablation_effects = {(l, h): [] for l, h in candidate_heads}
    mlp_ablation_effects = {m: [] for m in candidate_mlps}

    for prompt in test_prompts:
        baseline = get_sentiment_logit_diff(prompt, positive_tokens, negative_tokens)
        
        # Head ablations
        for layer, head in candidate_heads:
            ablated = ablate_head_and_measure(prompt, layer, head, positive_tokens, negative_tokens)
            head_ablation_effects[(layer, head)].append(ablated - baseline)
        
        # MLP ablations
        for layer in candidate_mlps:
            ablated = ablate_mlp_and_measure(prompt, layer, positive_tokens, negative_tokens)
            mlp_ablation_effects[layer].append(ablated - baseline)

    # Average effects
    print("\nAverage ablation effects (positive = reduces sarcasm detection):")
    print("\nAttention Heads:")
    head_avg_effects = {}
    for (layer, head), effects in head_ablation_effects.items():
        avg = np.mean(effects)
        head_avg_effects[(layer, head)] = avg
        if abs(avg) > 0.05:
            print(f"  a{layer}.h{head}: {avg:+.4f}")

    print("\nMLPs:")
    mlp_avg_effects = {}
    for layer, effects in mlp_ablation_effects.items():
        avg = np.mean(effects)
        mlp_avg_effects[layer] = avg
        print(f"  m{layer}: {avg:+.4f}")
    
    # Evaluation
    code_cells_info[22]['description'] = "Comprehensive ablation study"
    code_cells_info[22]['runnable'] = 'Y'
    code_cells_info[22]['correct_implementation'] = 'Y'
    code_cells_info[22]['redundant'] = 'N'
    code_cells_info[22]['irrelevant'] = 'N'
    print("\n✓ Block 22 executed successfully")
except Exception as e:
    code_cells_info[22]['runnable'] = 'N'
    code_cells_info[22]['error_note'] = str(e)
    print(f"\n✗ Block 22 failed: {e}")

BLOCK 22: Comprehensive ablation study
Comprehensive ablation study...



Average ablation effects (positive = reduces sarcasm detection):

Attention Heads:
  a9.h2: +0.1570
  a6.h7: +0.4864
  a7.h5: +0.0704
  a6.h3: +0.0814
  a8.h5: +0.0563
  a10.h4: -0.1828
  a6.h4: -0.0535

MLPs:
  m6: +0.0363
  m7: +0.1712
  m8: +0.0966
  m9: +0.3382
  m10: +0.5497

✓ Block 22 executed successfully


In [10]:
# Block 23: Define the minimal circuit
print("=" * 60)
print("BLOCK 23: Define the minimal circuit")
print("=" * 60)

try:
    # Threshold: include components with avg ablation effect > 0.1
    circuit_heads = []
    for (layer, head), effect in sorted(head_avg_effects.items(), key=lambda x: -x[1]):
        if effect > 0.1:
            circuit_heads.append(f"a{layer}.h{head}")
            print(f"Including head a{layer}.h{head} (ablation effect: {effect:.4f})")

    circuit_mlps = []
    for layer, effect in sorted(mlp_avg_effects.items(), key=lambda x: -x[1]):
        if effect > 0.1:
            circuit_mlps.append(f"m{layer}")
            print(f"Including MLP m{layer} (ablation effect: {effect:.4f})")

    print("\n" + "=" * 50)
    print("MINIMAL SARCASM CIRCUIT:")
    print("=" * 50)
    print(f"Attention Heads: {circuit_heads}")
    print(f"MLPs: {circuit_mlps}")
    print(f"Total components: {len(circuit_heads) + len(circuit_mlps)}")
    
    # Evaluation
    code_cells_info[23]['description'] = "Define the minimal circuit"
    code_cells_info[23]['runnable'] = 'Y'
    code_cells_info[23]['correct_implementation'] = 'Y'
    code_cells_info[23]['redundant'] = 'N'
    code_cells_info[23]['irrelevant'] = 'N'
    print("\n✓ Block 23 executed successfully")
except Exception as e:
    code_cells_info[23]['runnable'] = 'N'
    code_cells_info[23]['error_note'] = str(e)
    print(f"\n✗ Block 23 failed: {e}")

BLOCK 23: Define the minimal circuit
Including head a6.h7 (ablation effect: 0.4864)
Including head a9.h2 (ablation effect: 0.1570)
Including MLP m10 (ablation effect: 0.5497)
Including MLP m9 (ablation effect: 0.3382)
Including MLP m7 (ablation effect: 0.1712)

MINIMAL SARCASM CIRCUIT:
Attention Heads: ['a6.h7', 'a9.h2']
MLPs: ['m10', 'm9', 'm7']
Total components: 5

✓ Block 23 executed successfully


In [11]:
# Block 24: First attempt at circuit ablation (has bug)
print("=" * 60)
print("BLOCK 24: First attempt at circuit ablation (has bug)")
print("=" * 60)

# This block has a bug in the original notebook - hook_obj parameter naming
# Let's check if it runs:
try:
    def ablate_circuit_and_measure_v1(prompt, heads, mlps, positive_tokens, negative_tokens):
        """Ablate all circuit components and measure the effect."""
        
        hooks = []
        
        # Add head ablation hooks
        for head_name in heads:
            layer = int(head_name.split('.')[0][1:])
            head = int(head_name.split('.')[1][1:])
            
            def make_head_hook(l, h):
                def hook(activation, hook_obj):
                    activation[0, :, h, :] = 0
                    return activation
                return hook
            
            hooks.append((f"blocks.{layer}.attn.hook_z", make_head_hook(layer, head)))
        
        # Add MLP ablation hooks
        for mlp_name in mlps:
            layer = int(mlp_name[1:])
            
            def make_mlp_hook(l):
                def hook(activation, hook_obj):
                    activation[0, :, :] = 0
                    return activation
                return hook
            
            hooks.append((f"blocks.{layer}.hook_mlp_out", make_mlp_hook(layer)))
        
        tokens = model.to_tokens(prompt)
        
        with torch.no_grad():
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        
        last_logits = ablated_logits[0, -1, :]
        pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
        neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
        
        return (pos_logit - neg_logit).item()

    # Test the function (actually executes fine)
    test_result = ablate_circuit_and_measure_v1(test_prompts[0], circuit_heads, circuit_mlps, positive_tokens, negative_tokens)
    print(f"Test result: {test_result:.4f}")
    
    # Evaluation - This block has a bug (truncated output in original) but the code runs
    code_cells_info[24]['description'] = "First attempt at circuit ablation"
    code_cells_info[24]['runnable'] = 'Y'
    code_cells_info[24]['correct_implementation'] = 'Y'  # Implementation is correct
    code_cells_info[24]['redundant'] = 'Y'  # This is redundant as it's corrected in block 25
    code_cells_info[24]['irrelevant'] = 'N'
    code_cells_info[24]['error_note'] = "This block is redundant - functionality is duplicated in block 25"
    print("\n✓ Block 24 executed successfully (but is redundant with block 25)")
except Exception as e:
    code_cells_info[24]['runnable'] = 'N'
    code_cells_info[24]['error_note'] = str(e)
    print(f"\n✗ Block 24 failed: {e}")

BLOCK 24: First attempt at circuit ablation (has bug)

✗ Block 24 failed: ablate_circuit_and_measure_v1.<locals>.make_head_hook.<locals>.hook() got an unexpected keyword argument 'hook'


In [12]:
# Update evaluation for block 24 - it failed due to hook signature issue
code_cells_info[24]['description'] = "First attempt at circuit ablation (buggy)"
code_cells_info[24]['runnable'] = 'N'
code_cells_info[24]['correct_implementation'] = 'N'
code_cells_info[24]['redundant'] = 'Y'  # Redundant because it's fixed in block 25
code_cells_info[24]['irrelevant'] = 'N'
code_cells_info[24]['error_note'] = "Hook function signature uses 'hook_obj' but TransformerLens passes 'hook' keyword argument"

print("Block 24 marked as: Runnable=N, Correct=N, Redundant=Y (fixed in block 25)")

Block 24 marked as: Runnable=N, Correct=N, Redundant=Y (fixed in block 25)


In [13]:
# Block 25: Fixed circuit ablation function
print("=" * 60)
print("BLOCK 25: Fixed circuit ablation function")
print("=" * 60)

try:
    def ablate_circuit_and_measure(prompt, heads, mlps, positive_tokens, negative_tokens):
        """Ablate all circuit components and measure the effect."""
        
        hooks = []
        
        # Add head ablation hooks
        for head_name in heads:
            layer = int(head_name.split('.')[0][1:])
            head = int(head_name.split('.')[1][1:])
            
            def make_head_hook(h):
                def hook(activation, hook):
                    activation[0, :, h, :] = 0
                    return activation
                return hook
            
            hooks.append((f"blocks.{layer}.attn.hook_z", make_head_hook(head)))
        
        # Add MLP ablation hooks
        for mlp_name in mlps:
            layer = int(mlp_name[1:])
            
            def make_mlp_hook():
                def hook(activation, hook):
                    activation[0, :, :] = 0
                    return activation
                return hook
            
            hooks.append((f"blocks.{layer}.hook_mlp_out", make_mlp_hook()))
        
        tokens = model.to_tokens(prompt)
        
        with torch.no_grad():
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        
        last_logits = ablated_logits[0, -1, :]
        pos_logit = torch.mean(torch.stack([last_logits[t] for t in positive_tokens]))
        neg_logit = torch.mean(torch.stack([last_logits[t] for t in negative_tokens]))
        
        return (pos_logit - neg_logit).item()

    # Test circuit ablation
    print("Circuit ablation validation:")
    print("=" * 70)

    for prompt in test_prompts:
        baseline = get_sentiment_logit_diff(prompt, positive_tokens, negative_tokens)
        circuit_ablated = ablate_circuit_and_measure(
            prompt, circuit_heads, circuit_mlps, positive_tokens, negative_tokens
        )
        change = circuit_ablated - baseline
        
        print(f"\n'{prompt}'")
        print(f"  Baseline sentiment diff: {baseline:.4f}")
        print(f"  After circuit ablation: {circuit_ablated:.4f}")
        print(f"  Change: {change:+.4f} (more positive = sarcasm detection impaired)")
    
    # Evaluation
    code_cells_info[25]['description'] = "Fixed circuit ablation function"
    code_cells_info[25]['runnable'] = 'Y'
    code_cells_info[25]['correct_implementation'] = 'Y'
    code_cells_info[25]['redundant'] = 'N'
    code_cells_info[25]['irrelevant'] = 'N'
    print("\n✓ Block 25 executed successfully")
except Exception as e:
    code_cells_info[25]['runnable'] = 'N'
    code_cells_info[25]['error_note'] = str(e)
    print(f"\n✗ Block 25 failed: {e}")

BLOCK 25: Fixed circuit ablation function
Circuit ablation validation:

'Perfect, I lost my wallet. I feel'
  Baseline sentiment diff: -1.8634
  After circuit ablation: 0.2448
  Change: +2.1082 (more positive = sarcasm detection impaired)

'Oh great, another boring meeting. I feel'
  Baseline sentiment diff: -1.6365
  After circuit ablation: -0.3589
  Change: +1.2776 (more positive = sarcasm detection impaired)



'Wonderful, more tedious paperwork. I feel'
  Baseline sentiment diff: -1.2448
  After circuit ablation: -0.0887
  Change: +1.1560 (more positive = sarcasm detection impaired)

✓ Block 25 executed successfully


In [14]:
# Block 26: Test on literal sentences
print("=" * 60)
print("BLOCK 26: Test on literal sentences")
print("=" * 60)

try:
    literal_prompts = [
        "Perfect, I found some money. I feel",
        "Fantastic, the team won today. I feel", 
        "Oh great, a wonderful surprise. I feel",
    ]

    print("Testing on LITERAL sentences (should show less change):")
    print("=" * 70)

    for prompt in literal_prompts:
        baseline = get_sentiment_logit_diff(prompt, positive_tokens, negative_tokens)
        circuit_ablated = ablate_circuit_and_measure(
            prompt, circuit_heads, circuit_mlps, positive_tokens, negative_tokens
        )
        change = circuit_ablated - baseline
        
        print(f"\n'{prompt}'")
        print(f"  Baseline sentiment diff: {baseline:.4f}")
        print(f"  After circuit ablation: {circuit_ablated:.4f}")
        print(f"  Change: {change:+.4f}")
    
    # Evaluation
    code_cells_info[26]['description'] = "Test on literal sentences"
    code_cells_info[26]['runnable'] = 'Y'
    code_cells_info[26]['correct_implementation'] = 'Y'
    code_cells_info[26]['redundant'] = 'N'
    code_cells_info[26]['irrelevant'] = 'N'
    print("\n✓ Block 26 executed successfully")
except Exception as e:
    code_cells_info[26]['runnable'] = 'N'
    code_cells_info[26]['error_note'] = str(e)
    print(f"\n✗ Block 26 failed: {e}")

BLOCK 26: Test on literal sentences
Testing on LITERAL sentences (should show less change):

'Perfect, I found some money. I feel'
  Baseline sentiment diff: 0.2880
  After circuit ablation: 0.8094
  Change: +0.5213



'Fantastic, the team won today. I feel'
  Baseline sentiment diff: 1.8477
  After circuit ablation: 1.7991
  Change: -0.0486

'Oh great, a wonderful surprise. I feel'
  Baseline sentiment diff: 1.2687
  After circuit ablation: 1.0602
  Change: -0.2085

✓ Block 26 executed successfully
